# Instructions

1. Set your ALPHA_VANTAGE_API_KEY and FRED_API_KEY in the .env file  
    ! The script uses a lot of API Calls and Alpha Vantage has a cap of 25 API Calls per Day 

2. Install the required packages

3. Run the whole script or each section separately

At the end of each Section, the data gets merged into the "data" dataframe.  
So you can run only the first "Finance" Section and run the "Write to .csv" Section afterwards to get basic ohlcv Data and Technical Indicators.  
Dont forget to use the "Drop Na" Section if needed.  

### Sections:
Finance  
Can run independently

Target  
Needs Finance Data from Finance Section first

Calculations  
Needs Finance Data from Finance Section first

Lagged Features  
Needs Finance Data from Finance Section first
Needs calculated Data from Calculations Section first

Macroeconomic  
Can run independently

Drop Na  
Drops all missing values


# Setup

In [216]:
from dotenv import load_dotenv
import os
import requests
import pandas as pd
import io
import numpy as np
from fredapi import Fred
import yfinance as yf
import quantreo.features_engineering as fe

In [217]:
load_dotenv()

if os.getenv('ALPHA_VANTAGE_PREMIUM_API_KEY'):
    print("✓ Alpha Vantage Premium API KEY loaded successfully")
    premiumapikey = os.getenv('ALPHA_VANTAGE_PREMIUM_API_KEY')

elif os.getenv('ALPHA_VANTAGE_API_KEY'):
    print("✓ Alpha Vantage API KEY loaded successfully")
    apikey = os.getenv('ALPHA_VANTAGE_API_KEY')

else:
    print("✗ Error: Alpha Vantage API key not found")

if os.getenv('FRED_API_KEY'):
    print("✓ FRED API KEY loaded successfully")
    fredapikey = os.getenv('FRED_API_KEY')
else:
    print("✗ Error: FRED API key not found")


✓ Alpha Vantage Premium API KEY loaded successfully
✓ FRED API KEY loaded successfully


In [218]:
if premiumapikey:
    alphaapikey = premiumapikey
    print("✓ Alpha Vantage Premium API KEY set successfully")
else:
    alphaapikey = apikey
    print("✓ Alpha Vantage API KEY set successfully")

✓ Alpha Vantage Premium API KEY set successfully


# Finance Data Alpha Vantage API Calls 

In [219]:
baseUrl= "https://www.alphavantage.co/query?"

# symbols = ["0QKI.LON","0QLR.LON","NSRGY","RHO6.FRK","ABBNY","UBS","0QP2.LON","0QKY.LON","0QNO.LON","0QPS.LON","0A0D.LON","0Z4C.LON","0Q0Q.LON","0QMG.LON","AMRZ","0QQ2.LON","0QMW","0QK6"]
# print("Symbols: ",len(symbols))


symbol = "UBS"
interval = "daily" # Maybe not needed anymore
datatype = "csv"


## Daily

In [220]:
function = "TIME_SERIES_DAILY" #TIME_SERIES_DAILY
adjusted = "true"
extended_hours = "true"
outputsize = "full"

url = (
    f"{baseUrl}"
    f"function={function}&symbol={symbol}"
    f"&outputsize={outputsize}&datatype={datatype}"
    f"&apikey={alphaapikey}"
)

response = requests.get(url)
print("Status Code:",response.status_code)
print("Raw CSV Data:")
print(response.text[:300])

df1 = pd.read_csv(io.StringIO(response.text))
df1["timestamp"] = pd.to_datetime(df1["timestamp"])
df1 = df1.set_index("timestamp")
print(df1.head())
df1.head()
print(df1.sort_index(ascending=True))

Status Code: 200
Raw CSV Data:
timestamp,open,high,low,close,volume
2025-11-26,38.0500,38.1950,37.9850,38.0500,1192606
2025-11-25,37.2900,37.6400,37.0950,37.5900,1350342
2025-11-24,36.6100,36.8950,36.4435,36.7900,1973760
2025-11-21,37.0400,37.2000,36.6550,37.0700,1989950
2025-11-20,37.7100,38.1400,36.7800,36.7900,2701233
20
             open    high      low  close   volume
timestamp                                         
2025-11-26  38.05  38.195  37.9850  38.05  1192606
2025-11-25  37.29  37.640  37.0950  37.59  1350342
2025-11-24  36.61  36.895  36.4435  36.79  1973760
2025-11-21  37.04  37.200  36.6550  37.07  1989950
2025-11-20  37.71  38.140  36.7800  36.79  2701233
             open    high      low    close   volume
timestamp                                           
2014-11-21  17.47  17.470  17.3900  17.3900     7000
2014-11-24  17.55  17.990  17.3800  17.5800     6832
2014-11-25  17.56  17.560  17.4835  17.4835     4426
2014-11-26  17.55  23.200  17.5200  23.2000     2817

## Technical Indicators

### SMA

In [ ]:
function= "SMA"
interval = "daily" 
#time_periods= ["6","24","72"]
time_period= "6"
series_type= "close"


url = (f"{baseUrl}"
        f"function={function}&symbol={symbol}&interval={interval}"
        f"&time_period={time_period}&series_type={series_type}"
        f"&datatype={datatype}&apikey={alphaapikey}")

response = requests.get(url)
print("Status Code:",response.status_code)
print("Raw CSV Data:")
print(response.text[:100])

df2 = pd.read_csv(io.StringIO(response.text))
df2 = df2.rename(columns={"time": "timestamp"})
df2["timestamp"] = pd.to_datetime(df2["timestamp"])
df2 = df2.set_index("timestamp")
print(df2.head())
print(df2.sort_index(ascending=True).head())


Status Code: 200
Raw CSV Data:
time,SMA
2025-11-26,37.4150
2025-11-25,37.4117
2025-11-24,37.5217
2025-11-21,37.8767
2025-11-20
                SMA
timestamp          
2025-11-26  37.4150
2025-11-25  37.4117
2025-11-24  37.5217
2025-11-21  37.8767
2025-11-20  38.2100
                SMA
timestamp          
2014-12-01  13.0558
2014-12-02  13.1272
2014-12-03  13.1892
2014-12-04  13.2684
2014-12-05  12.6820


### EMA

In [ ]:
function= "EMA"
interval = "daily" 
#time_periods= ["6","24"]
time_period= "6"
series_type= "close"


url = (f"{baseUrl}"
        f"function={function}&symbol={symbol}&interval={interval}"
        f"&time_period={time_period}&series_type={series_type}"
        f"&datatype={datatype}&apikey={alphaapikey}")

response = requests.get(url)
print("Status Code:",response.status_code)
print("Raw CSV Data:")
print(response.text[:100])

df3 = pd.read_csv(io.StringIO(response.text))
df3 = df3.rename(columns={"time": "timestamp"})
df3["timestamp"] = pd.to_datetime(df3["timestamp"])
df3 = df3.set_index("timestamp")
print(df3.head())
print(df3.sort_index(ascending=True).head())

Status Code: 200
Raw CSV Data:
time,EMA
2025-11-26,37.6460
2025-11-25,37.4844
2025-11-24,37.4421
2025-11-21,37.7030
2025-11-20
                EMA
timestamp          
2025-11-26  37.6460
2025-11-25  37.4844
2025-11-24  37.4421
2025-11-21  37.7030
2025-11-20  37.9561
                EMA
timestamp          
2014-12-01  13.0558
2014-12-02  12.9373
2014-12-03  12.8747
2014-12-04  12.8400
2014-12-05  12.8213


### RSI

In [ ]:
function= "RSI"
interval = "daily" 
#time_periods= ["14"]
time_period= "14"
series_type= "close"


url = (f"{baseUrl}"
        f"function={function}&symbol={symbol}&interval={interval}"
        f"&time_period={time_period}&series_type={series_type}"
        f"&datatype={datatype}&apikey={alphaapikey}")

response = requests.get(url)
print("Status Code:",response.status_code)
print("Raw CSV Data:")
print(response.text[:100])

df4 = pd.read_csv(io.StringIO(response.text))
df4 = df4.rename(columns={"time": "timestamp"})
df4["timestamp"] = pd.to_datetime(df4["timestamp"])
df4 = df4.set_index("timestamp")
print(df4.head())
print(df4.sort_index(ascending=True).head())

Status Code: 200
Raw CSV Data:
time,RSI
2025-11-26,47.8205
2025-11-25,43.7847
2025-11-24,35.7607
2025-11-21,37.5003
2025-11-20
                RSI
timestamp          
2025-11-26  47.8205
2025-11-25  43.7847
2025-11-24  35.7607
2025-11-21  37.5003
2025-11-20  34.5436
                RSI
timestamp          
2014-12-12  50.0401
2014-12-15  49.1066
2014-12-16  49.2918
2014-12-17  50.0263
2014-12-18  51.1937


### ATR

In [ ]:
function= "ATR"
interval = "daily" 
#time_periods= ["14"]
time_period= "14"


url = (f"{baseUrl}"
        f"function={function}&symbol={symbol}&interval={interval}"
        f"&time_period={time_period}"
        f"&datatype={datatype}&apikey={alphaapikey}")

response = requests.get(url)
print("Status Code:",response.status_code)
print("Raw CSV Data:")
print(response.text[:100])

df5 = pd.read_csv(io.StringIO(response.text))
df5 = df5.rename(columns={"time": "timestamp"})
df5["timestamp"] = pd.to_datetime(df5["timestamp"])
df5 = df5.set_index("timestamp")
print(df5.head())
print(df5.sort_index(ascending=True).head())

Status Code: 200
Raw CSV Data:
time,ATR
2025-11-26,0.6841
2025-11-25,0.6902
2025-11-24,0.6779
2025-11-21,0.6819
2025-11-20,0.6
               ATR
timestamp         
2025-11-26  0.6841
2025-11-25  0.6902
2025-11-24  0.6779
2025-11-21  0.6819
2025-11-20  0.6924
               ATR
timestamp         
2014-12-12  0.6952
2014-12-15  0.6651
2014-12-16  0.6366
2014-12-17  0.6052
2014-12-18  0.5740


### BBANDS

In [ ]:
function= "BBANDS"
interval = "daily" 
#time_periods= ["20"]
time_period= "20"
series_type= "close"
nbdevup= "2"
nbdevdn= "2"


url = (f"{baseUrl}"
        f"function={function}&symbol={symbol}&interval={interval}"
        f"&time_period={time_period}&series_type={series_type}"
        f"&datatype={datatype}&apikey={alphaapikey}"
        f"&nbdevup={nbdevup}&nbdevdn={nbdevdn}")

response = requests.get(url)
print("Status Code:",response.status_code)
print("Raw CSV Data:")
print(response.text[:100])

df6 = pd.read_csv(io.StringIO(response.text))
df6 = df6.rename(columns={"time": "timestamp"})
df6["timestamp"] = pd.to_datetime(df6["timestamp"])
df6 = df6.set_index("timestamp")
print(df6.head())
print(df6.sort_index(ascending=True).head())

Status Code: 200
Raw CSV Data:
time,Real Lower Band,Real Middle Band,Real Upper Band
2025-11-26,36.7377,38.1360,39.5343
2025-11-2
            Real Lower Band  Real Middle Band  Real Upper Band
timestamp                                                     
2025-11-26          36.7377           38.1360          39.5343
2025-11-25          36.7484           38.1475          39.5466
2025-11-24          36.7906           38.2225          39.6544
2025-11-21          37.0340           38.3130          39.5920
2025-11-20          37.2051           38.3605          39.5159
            Real Lower Band  Real Middle Band  Real Upper Band
timestamp                                                     
2014-12-19          10.8880           12.6271          14.3662
2014-12-22          10.8971           12.6320          14.3670
2014-12-23          10.8922           12.6292          14.3662
2014-12-24          10.8998           12.6336          14.3675
2014-12-26          12.0066           12.4395      

## df merge

In [226]:
dfs = [df1, df2, df3, df4, df5, df6]

df_finance = dfs[0]

for df in dfs[1:]:
    df_finance = df_finance.merge(df, on="timestamp", how="left")

df_finance.head()

,open,high,low,close,volume,SMA,EMA,RSI,ATR,Real Lower Band,Real Middle Band,Real Upper Band
timestamp,,,,,,,,,,,,
2025-11-26,38.05,38.195,37.9850,38.05,1192606,37.4150,37.6460,47.8205,0.6841,36.7377,38.1360,39.5343
2025-11-25,37.29,37.640,37.0950,37.59,1350342,37.4117,37.4844,43.7847,0.6902,36.7484,38.1475,39.5466
2025-11-24,36.61,36.895,36.4435,36.79,1973760,37.5217,37.4421,35.7607,0.6779,36.7906,38.2225,39.6544
2025-11-21,37.04,37.200,36.6550,37.07,1989950,37.8767,37.7030,37.5003,0.6819,37.0340,38.3130,39.5920
2025-11-20,37.71,38.140,36.7800,36.79,2701233,38.2100,37.9561,34.5436,0.6924,37.2051,38.3605,39.5159


## Data Checks

In [227]:
#df_finance["timestamp"] = pd.to_datetime(df_finance["timestamp"])
#df_finance = df_finance.set_index("timestamp")
df = df_finance.sort_index(ascending=True)
print(df.isna().sum().sum())

# for d in dfs:
#     print(d.isna().sum().sum())
#     print(len(d))
#     print(len(d.dropna()))

print(df_finance.dropna().sort_index(ascending=True).head())
print(df_finance.dropna().sort_index(ascending=False).head())




95
             open     high     low  close   volume      SMA      EMA      RSI  \
timestamp                                                                       
2014-12-19  17.33  17.4466  17.270  17.38  1021196  12.1904  12.2663  49.9908   
2014-12-22  17.45  17.5400  17.445  17.53   701889  12.2056  12.2791  50.8390   
2014-12-23  17.50  17.6100  17.470  17.50  1097125  12.2431  12.2822  50.6540   
2014-12-24  17.47  17.6300  17.460  17.61   450226  12.2887  12.3064  51.3532   
2014-12-26  17.52  17.7200  17.510  17.67   531820  12.3238  12.3358  51.7547   

               ATR  Real Lower Band  Real Middle Band  Real Upper Band  
timestamp                                                               
2014-12-19  0.5496          10.8880           12.6271          14.3662  
2014-12-22  0.5183          10.8971           12.6320          14.3670  
2014-12-23  0.4883          10.8922           12.6292          14.3662  
2014-12-24  0.4620          10.8998           12.6336          1

## Data merge

In [228]:
data = df_finance.sort_index(ascending=True)

In [229]:
# Dataframe Preparation or Reset
df_target = df_finance.sort_index(ascending=True)
df_calc = df_finance.sort_index(ascending=True)
df_macro = df_finance.sort_index(ascending=True)

# Target

https://docs.quantreo.com/features-engineering/volatility/

## CTC Volatility

In [230]:
df_target["ctc_vol"] = fe.volatility.close_to_close_volatility(df=df_target, close_col="close", window_size=5)

## Parkinson Volatility

In [231]:
df_target["parkinson_vol"] = fe.volatility.parkinson_volatility(df=df_target, high_col="high", low_col="low", window_size=5)

## Target Shift

In [232]:
df_target["ctc_vol_shifted"] = df_target["ctc_vol"].shift(-1)

In [233]:
df_target["parkinson_vol_shifted"] = df_target["parkinson_vol"].shift(-1)

In [234]:
#df_target["target_vol_next_1d"] = df_target["range_t"].shift(-1)

In [235]:
# df_target["target_vol_next"] = df_target["abs_log_return"].shift(-1)
# df_target["target_vol_next"] = df_target["ret_vol_6"].shift(-1) # For trading this is not so good as abs_log_return

In [236]:
# df_target["target_vol_next_3d"] = df_target["range_t"].shift(-1).rolling(3).mean()
# df_target["target_vol_next_5d"] = df_target["range_t"].shift(-1).rolling(5).mean()

## Data merge

In [237]:
data = data.drop(columns=df_target.columns.intersection(data.columns))

data = data.merge(df_target, left_index=True, right_index=True, how="left")
data.sort_index(ascending=True).head()

,open,high,low,close,volume,SMA,EMA,RSI,ATR,Real Lower Band,Real Middle Band,Real Upper Band,ctc_vol,parkinson_vol,ctc_vol_shifted,parkinson_vol_shifted
timestamp,,,,,,,,,,,,,,,,
2014-11-21,17.47,17.47,17.3900,17.3900,7000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2014-11-24,17.55,17.99,17.3800,17.5800,6832,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2014-11-25,17.56,17.56,17.4835,17.4835,4426,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2014-11-26,17.55,23.20,17.5200,23.2000,2817,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2014-11-28,18.09,18.13,17.9600,17.9900,111605,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.190212,0.076048


# Calculations

In [238]:
df_finance.head()

,open,high,low,close,volume,SMA,EMA,RSI,ATR,Real Lower Band,Real Middle Band,Real Upper Band
timestamp,,,,,,,,,,,,
2025-11-26,38.05,38.195,37.9850,38.05,1192606,37.4150,37.6460,47.8205,0.6841,36.7377,38.1360,39.5343
2025-11-25,37.29,37.640,37.0950,37.59,1350342,37.4117,37.4844,43.7847,0.6902,36.7484,38.1475,39.5466
2025-11-24,36.61,36.895,36.4435,36.79,1973760,37.5217,37.4421,35.7607,0.6779,36.7906,38.2225,39.6544
2025-11-21,37.04,37.200,36.6550,37.07,1989950,37.8767,37.7030,37.5003,0.6819,37.0340,38.3130,39.5920
2025-11-20,37.71,38.140,36.7800,36.79,2701233,38.2100,37.9561,34.5436,0.6924,37.2051,38.3605,39.5159


In [239]:
df_finance.columns

Index(['open', 'high', 'low', 'close', 'volume', 'SMA', 'EMA', 'RSI', 'ATR',
       'Real Lower Band', 'Real Middle Band', 'Real Upper Band'],
      dtype='object')

## Simple Return

In [240]:
df_calc['return'] = df_calc['close'].pct_change() # Maybe only use Log Return and delete this. They are to similar

## Log Return

In [241]:
df_calc["log_return"] = np.log(df_calc["close"] / df_calc["close"].shift(1))

## Absolute Log Return

In [242]:
df_calc["abs_log_return"] = df_calc["log_return"].abs()
df_calc.head()

,open,high,low,close,volume,SMA,EMA,RSI,ATR,Real Lower Band,Real Middle Band,Real Upper Band,return,log_return,abs_log_return
timestamp,,,,,,,,,,,,,,,
2014-11-21,17.47,17.47,17.3900,17.3900,7000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2014-11-24,17.55,17.99,17.3800,17.5800,6832,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.010926,0.010867,0.010867
2014-11-25,17.56,17.56,17.4835,17.4835,4426,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.005489,-0.005504,0.005504
2014-11-26,17.55,23.20,17.5200,23.2000,2817,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.326965,0.282895,0.282895
2014-11-28,18.09,18.13,17.9600,17.9900,111605,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.224569,-0.254336,0.254336


## Current candle range

In [243]:
df_calc["range_t"] = np.log(df_calc["high"] / df_calc["low"])

## Candle Shape

In [244]:
df_calc["body"] = (df_calc["close"] - df_calc["open"]).abs()

df_calc["upper_wick"] = df_calc["high"] - df_calc[["open", "close"]].max(axis=1)
df_calc["lower_wick"] = df_calc[["open", "close"]].min(axis=1) - df_calc["low"]

In [245]:
eps = 1e-9 # Safeguard so we never divide by 0
range_ = df_calc["high"] - df_calc["low"]

df_calc["body_ratio"] = df_calc["body"] / (range_ + eps)
df_calc["upper_wick_ratio"] = df_calc["upper_wick"] / (range_ + eps)
df_calc["lower_wick_ratio"] = df_calc["lower_wick"] / (range_ + eps)
print(1+eps)

1.000000001


## Bollinger Bands Width

In [246]:
eps = 1e-9

df_calc["bb_width"] = df_calc["Real Upper Band"] - df_calc["Real Lower Band"]
df_calc["bb_width_norm"] = df_calc["bb_width"] / (df_calc["Real Middle Band"] + eps)

## Time-based

In [247]:
df_calc["day_of_week"] = df_calc.index.dayofweek

## Indicator Delta

In [248]:
# df_calc["RSI_delta"] = df_calc["RSI"] - df_calc["RSI"].shift(1)
# df_calc["ATR_delta"] = df_calc["ATR"] - df_calc["ATR"].shift(1)
# df_calc["bb_width_chg"] = df_calc["bb_width_norm"] - df_calc["bb_width_norm"].shift(1)

## Data merge

In [249]:
data = data.drop(columns=df_calc.columns.intersection(data.columns))

data = data.merge(df_calc, left_index=True, right_index=True, how="left")
data.sort_index(ascending=False).head()

,ctc_vol,parkinson_vol,ctc_vol_shifted,parkinson_vol_shifted,open,high,low,close,volume,SMA,...,range_t,body,upper_wick,lower_wick,body_ratio,upper_wick_ratio,lower_wick_ratio,bb_width,bb_width_norm,day_of_week
timestamp,,,,,,,,,,,,,,,,,,,,,
2025-11-26,0.023114,0.011973,NaN,NaN,38.05,38.195,37.9850,38.05,1192606,37.4150,...,0.005513,0.00,0.145,0.0650,0.000000,0.690476,0.309524,2.7966,0.073332,2
2025-11-25,0.022277,0.011813,0.023114,0.011973,37.29,37.640,37.0950,37.59,1350342,37.4117,...,0.014585,0.30,0.050,0.1950,0.550459,0.091743,0.357798,2.7982,0.073352,1
2025-11-24,0.017886,0.012564,0.022277,0.011813,36.61,36.895,36.4435,36.79,1973760,37.5217,...,0.012313,0.18,0.105,0.1665,0.398671,0.232558,0.368771,2.8638,0.074924,0
2025-11-21,0.018387,0.012516,0.017886,0.012564,37.04,37.200,36.6550,37.07,1989950,37.8767,...,0.014759,0.03,0.130,0.3850,0.055046,0.238532,0.706422,2.5580,0.066766,4
2025-11-20,0.016286,0.009474,0.018387,0.012516,37.71,38.140,36.7800,36.79,2701233,38.2100,...,0.036309,0.92,0.430,0.0100,0.676471,0.316176,0.007353,2.3108,0.060239,3


In [250]:
# The most current row gets dropped cause of the target value shift (target_vol_next_1d)


first_row = data.sort_index(ascending=False).iloc[0]
na_columns = first_row[first_row.isna()].index

print("Date of first row:", data.sort_index(ascending=False).index[0])
print("Columns with NA in first row:", na_columns.tolist())


Date of first row: 2025-11-26 00:00:00
Columns with NA in first row: ['ctc_vol_shifted', 'parkinson_vol_shifted']


# Lagged Features

In [523]:
df_lag = df_calc.sort_index(ascending=True)

## Rolling Volatility

In [591]:
windows = [6,24]

for w in windows:
    df_lag[f'range_mean_{w}'] = df_lag["range_t"].rolling(w).mean()
    df_lag[f'range_std_{w}'] = df_lag["range_t"].rolling(w).std()

## Lagged OHLC(V)

In [592]:
lags = 3

for col in ['open', 'high', 'low', 'close']:
    for lag in range(1, lags + 1):
        df_lag[f'{col}_lag{lag}'] = df_lag[col].shift(lag)

## Lagged Log Return

In [593]:
lags = 3

for lag in range(1, lags + 1):
    df_lag[f'log_return_lag{lag}'] = df_lag["log_return"].shift(lag)

## Rolling Volatility of Log Returns

In [594]:
windows = [3,6,12,24,72]

for w in windows:
    df_lag[f'ret_vol_{w}'] = df_lag["log_return"].rolling(w).std()

## Realized Variance

In [595]:
df_lag["rv_6"] = (df_lag["log_return"]**2).rolling(6).sum()

## Lagged Absolute Log Return

In [529]:
lags = 3

for lag in range(1, lags + 1):
    df_lag[f'abs_log_return_lag{lag}'] = df_lag["abs_log_return"].shift(lag)

## Lagged Rolling Volatility of Returns

In [530]:
lags = 3

ret_vol_cols = [c for c in df_lag.columns if c.startswith("ret_vol_")]

for col in ret_vol_cols:
    for lag in range(1, lags + 1):
        df_lag[f'{col}_lag{lag}'] = df_lag[col].shift(lag)

## Lagged Realized Variance

In [531]:
lags = 3

for lag in range(1, lags + 1):
    df_lag[f'rv_6_lag{lag}'] = df_lag["rv_6"].shift(lag)

## Lagged Candle Range

In [532]:
lags = 3

for lag in range(1, lags + 1):
    df_lag[f'range_t_lag{lag}'] = df_lag["range_t"].shift(lag)

## Lagged Candle Shape

In [533]:
lags = 3

for lag in range(1, lags + 1):
    df_lag[f'body_ratio_lag{lag}'] = df_lag["body_ratio"].shift(lag)

## Rolling Volume

In [534]:
eps = 1e-9

df_lag["vol_SMA_6"] = df_lag["volume"].rolling(6).mean()
df_lag["vol_SMA_24"] = df_lag["volume"].rolling(24).mean()
df_lag["rel_volume"] = df_lag["volume"] / (df_lag["vol_SMA_24"] + eps)

## Lagged Volume

In [535]:
lags = 3

for lag in range(1, lags + 1):
    df_lag[f'rel_volume_lag{lag}'] = df_lag["rel_volume"].shift(lag)

## Lagged SMA

In [536]:
lags = 3

for lag in range(1, lags + 1):
    df_lag[f'SMA_lag{lag}'] = df_lag["SMA"].shift(lag)

## Lagged EMA

In [537]:
lags = 3

for lag in range(1, lags + 1):
    df_lag[f'EMA_lag{lag}'] = df_lag["EMA"].shift(lag)

## Lagged RSI

In [538]:
lags = 3

for lag in range(1, lags + 1):
    df_lag[f'RSI_lag{lag}'] = df_lag["RSI"].shift(lag)

## Lagged ATR

In [539]:
lags = 3

for lag in range(1, lags + 1):
    df_lag[f'ATR_lag{lag}'] = df_lag["ATR"].shift(lag)

## Lagged Bollinger Bands Width

In [540]:
lags = 3

for lag in range(1, lags + 1):
    df_lag[f'bb_width_norm_lag{lag}'] = df_lag["bb_width_norm"].shift(lag)

## Data merge

In [596]:
data = data.drop(columns=df_lag.columns.intersection(data.columns))

data = data.merge(df_lag, left_index=True, right_index=True, how="left")
data.sort_index(ascending=False).head()

,ctc_vol,parkinson_vol,ctc_vol_shifted,open,high,low,close,volume,SMA,EMA,...,EMA_lag3,RSI_lag1,RSI_lag2,RSI_lag3,ATR_lag1,ATR_lag2,ATR_lag3,bb_width_norm_lag1,bb_width_norm_lag2,bb_width_norm_lag3
timestamp,,,,,,,,,,,,,,,,,,,,,
2025-11-26,0.023114,0.011973,NaN,38.05,38.195,37.9850,38.05,1192606,37.4150,37.6460,...,37.7030,43.7847,35.7607,37.5003,0.6902,0.6779,0.6819,0.073352,0.074924,0.066766
2025-11-25,0.022277,0.011813,0.023114,37.29,37.640,37.0950,37.59,1350342,37.4117,37.4844,...,37.9561,35.7607,37.5003,34.5436,0.6779,0.6819,0.6924,0.074924,0.066766,0.060239
2025-11-24,0.017886,0.012564,0.022277,36.61,36.895,36.4435,36.79,1973760,37.5217,37.4421,...,38.4226,37.5003,34.5436,44.3556,0.6819,0.6924,0.6364,0.066766,0.060239,0.047273
2025-11-21,0.018387,0.012516,0.017886,37.04,37.200,36.6550,37.07,1989950,37.8767,37.7030,...,38.5116,34.5436,44.3556,42.5280,0.6924,0.6364,0.6581,0.060239,0.047273,0.047755
2025-11-20,0.016286,0.009474,0.018387,37.71,38.140,36.7800,36.79,2701233,38.2100,37.9561,...,38.7043,44.3556,42.5280,44.2755,0.6364,0.6581,0.6687,0.047273,0.047755,0.050977


In [597]:
# The most current row gets dropped cause of the target value shift (target_vol_next_1d)


first_row = data.sort_index(ascending=False).iloc[0]
na_columns = first_row[first_row.isna()].index

print("Date of first row:", data.sort_index(ascending=False).index[0])
print("Columns with NA in first row:", na_columns.tolist())


Date of first row: 2025-11-26 00:00:00
Columns with NA in first row: ['ctc_vol_shifted']


## Drop NaN 

In [598]:
data = data.dropna()
data = data.copy()
print(len(data))
print(data.columns)
data.sort_index(ascending=False).head()

2694
Index(['ctc_vol', 'parkinson_vol', 'ctc_vol_shifted', 'open', 'high', 'low',
       'close', 'volume', 'SMA', 'EMA',
       ...
       'EMA_lag3', 'RSI_lag1', 'RSI_lag2', 'RSI_lag3', 'ATR_lag1', 'ATR_lag2',
       'ATR_lag3', 'bb_width_norm_lag1', 'bb_width_norm_lag2',
       'bb_width_norm_lag3'],
      dtype='object', length=101)


,ctc_vol,parkinson_vol,ctc_vol_shifted,open,high,low,close,volume,SMA,EMA,...,EMA_lag3,RSI_lag1,RSI_lag2,RSI_lag3,ATR_lag1,ATR_lag2,ATR_lag3,bb_width_norm_lag1,bb_width_norm_lag2,bb_width_norm_lag3
timestamp,,,,,,,,,,,,,,,,,,,,,
2025-11-25,0.022277,0.011813,0.023114,37.29,37.640,37.0950,37.59,1350342,37.4117,37.4844,...,37.9561,35.7607,37.5003,34.5436,0.6779,0.6819,0.6924,0.074924,0.066766,0.060239
2025-11-24,0.017886,0.012564,0.022277,36.61,36.895,36.4435,36.79,1973760,37.5217,37.4421,...,38.4226,37.5003,34.5436,44.3556,0.6819,0.6924,0.6364,0.066766,0.060239,0.047273
2025-11-21,0.018387,0.012516,0.017886,37.04,37.200,36.6550,37.07,1989950,37.8767,37.7030,...,38.5116,34.5436,44.3556,42.5280,0.6924,0.6364,0.6581,0.060239,0.047273,0.047755
2025-11-20,0.016286,0.009474,0.018387,37.71,38.140,36.7800,36.79,2701233,38.2100,37.9561,...,38.7043,44.3556,42.5280,44.2755,0.6364,0.6581,0.6687,0.047273,0.047755,0.050977
2025-11-19,0.008543,0.009376,0.016286,38.11,38.265,37.9100,38.20,1307518,38.6783,38.4226,...,38.8860,42.5280,44.2755,50.0969,0.6581,0.6687,0.6578,0.047755,0.050977,0.050855


In [599]:
data.head()

,ctc_vol,parkinson_vol,ctc_vol_shifted,open,high,low,close,volume,SMA,EMA,...,EMA_lag3,RSI_lag1,RSI_lag2,RSI_lag3,ATR_lag1,ATR_lag2,ATR_lag3,bb_width_norm_lag1,bb_width_norm_lag2,bb_width_norm_lag3
timestamp,,,,,,,,,,,,,,,,,,,,,
2015-03-13,0.011052,0.006423,0.012903,17.41,17.6200,17.35,17.60,1311742,12.2220,12.2425,...,12.2498,51.0481,46.1463,48.1355,0.1837,0.1875,0.1939,0.036058,0.056490,0.059970
2015-03-16,0.012903,0.006241,0.011185,17.90,17.9800,17.86,17.91,1542476,12.2653,12.3383,...,12.1970,56.7064,51.0481,46.1463,0.1841,0.1837,0.1875,0.029574,0.036058,0.056490
2015-03-17,0.011185,0.006014,0.012623,17.72,17.8300,17.66,17.78,1415377,12.3004,12.3807,...,12.1955,62.7033,56.7064,51.0481,0.1900,0.1841,0.1837,0.035202,0.029574,0.036058
2015-03-18,0.012623,0.006051,0.015413,17.91,18.3300,17.89,18.27,2022231,12.4186,12.5092,...,12.2425,59.0117,62.7033,56.7064,0.1890,0.1900,0.1841,0.038003,0.035202,0.029574
2015-03-19,0.015413,0.008586,0.018285,18.18,18.2601,18.11,18.14,1171222,12.5310,12.5750,...,12.3383,66.9176,59.0117,62.7033,0.2031,0.1890,0.1900,0.053834,0.038003,0.035202


# Macroeconomic Data

## Alpha Vantage API Calls

### Federal Funds

In [55]:
function= "FEDERAL_FUNDS_RATE"

interval = "daily"

url = (f"{baseUrl}"
        f"function={function}&interval={interval}&datatype={datatype}&apikey={alphaapikey}")

response = requests.get(url)
print("Status Code:",response.status_code)
print("Raw CSV Data:")
print(response.text[:100])

df1_macro = pd.read_csv(io.StringIO(response.text))
df1_macro = df1_macro.rename(columns={"time": "timestamp"})
df1_macro = df1_macro.rename(columns={"value": function})
df1_macro["timestamp"] = pd.to_datetime(df1_macro["timestamp"])
df1_macro = df1_macro.set_index("timestamp")
print(df1_macro.head())
print(df1_macro.sort_index(ascending=True).head())

Status Code: 200
Raw CSV Data:
timestamp,value
2025-11-20,3.88
2025-11-19,3.88
2025-11-18,3.88
2025-11-17,3.88
2025-11-16,3.88
            FEDERAL_FUNDS_RATE
timestamp                     
2025-11-20                3.88
2025-11-19                3.88
2025-11-18                3.88
2025-11-17                3.88
2025-11-16                3.88
            FEDERAL_FUNDS_RATE
timestamp                     
1954-07-01                1.13
1954-07-02                1.25
1954-07-03                1.25
1954-07-04                1.25
1954-07-05                0.88


### Inflation

In [56]:
function= "INFLATION"

url = (f"{baseUrl}"
        f"function={function}&datatype={datatype}&apikey={alphaapikey}")

response = requests.get(url)
print("Status Code:",response.status_code)
print("Raw CSV Data:")
print(response.text[:100])

df2_macro = pd.read_csv(io.StringIO(response.text))
df2_macro = df2_macro.rename(columns={"value": function})
df2_macro["timestamp"] = pd.to_datetime(df2_macro["timestamp"])
df2_macro = df2_macro.set_index("timestamp")
print(df2_macro.head())
print(df2_macro.sort_index(ascending=True).head())

# test = pd.DataFrame(index=df_finance.index)
# test = test.merge(df2_macro, on="timestamp", how="left")
# # test = test.ffill()
# test = test.interpolate(method="time")
# test.head(479)

Status Code: 200
Raw CSV Data:
timestamp,value
2024-01-01,2.94952520485207
2023-01-01,4.11633838374488
2022-01-01,8.002799820521
            INFLATION
timestamp            
2024-01-01   2.949525
2023-01-01   4.116338
2022-01-01   8.002800
2021-01-01   4.697859
2020-01-01   1.233584
            INFLATION
timestamp            
1960-01-01   1.457976
1961-01-01   1.070724
1962-01-01   1.198773
1963-01-01   1.239669
1964-01-01   1.278912


### Real GDP

In [57]:
function= "REAL_GDP"

interval = "quarterly"

url = (f"{baseUrl}"
        f"function={function}&interval={interval}&datatype={datatype}&apikey={alphaapikey}")

response = requests.get(url)
print("Status Code:",response.status_code)
print("Raw CSV Data:")
print(response.text[:100])

df3_macro = pd.read_csv(io.StringIO(response.text))
df3_macro = df3_macro.rename(columns={"time": "timestamp"})
df3_macro = df3_macro.rename(columns={"value": function})
df3_macro["timestamp"] = pd.to_datetime(df3_macro["timestamp"])
df3_macro = df3_macro.set_index("timestamp")
print(df3_macro.head())
print(df3_macro.sort_index(ascending=True).head())

Status Code: 200
Raw CSV Data:
timestamp,value
2025-04-01,5943.384
2025-01-01,5776.724
2024-10-01,5997.184
2024-07-01,5884.567
            REAL_GDP
timestamp           
2025-04-01  5943.384
2025-01-01  5776.724
2024-10-01  5997.184
2024-07-01  5884.567
2024-04-01  5829.384
            REAL_GDP
timestamp           
2002-01-01  3501.118
2002-04-01  3608.496
2002-07-01  3650.253
2002-10-01  3712.845
2003-01-01  3582.767


### Real GDP per Capita

In [58]:
function= "REAL_GDP_PER_CAPITA"

url = (f"{baseUrl}"
        f"function={function}&datatype={datatype}&apikey={alphaapikey}")

response = requests.get(url)
print("Status Code:",response.status_code)
print("Raw CSV Data:")
print(response.text[:100])

df4_macro = pd.read_csv(io.StringIO(response.text))
df4_macro = df4_macro.rename(columns={"time": "timestamp"})
df4_macro = df4_macro.rename(columns={"value": function})
df4_macro["timestamp"] = pd.to_datetime(df4_macro["timestamp"])
df4_macro = df4_macro.set_index("timestamp")
print(df4_macro.head())
print(df4_macro.sort_index(ascending=True).head())

Status Code: 200
Raw CSV Data:
timestamp,value
2025-04-01,69499.0
2025-01-01,68937.0
2024-10-01,69136.0
2024-07-01,68926.0
202
            REAL_GDP_PER_CAPITA
timestamp                      
2025-04-01              69499.0
2025-01-01              68937.0
2024-10-01              69136.0
2024-07-01              68926.0
2024-04-01              68504.0
            REAL_GDP_PER_CAPITA
timestamp                      
1947-01-01              15248.0
1947-04-01              15139.0
1947-07-01              15039.0
1947-10-01              15204.0
1948-01-01              15371.0


### Treasury Yield

In [59]:
function= "TREASURY_YIELD"

interval = "daily"

url = (f"{baseUrl}"
        f"function={function}&interval={interval}&datatype={datatype}&apikey={alphaapikey}")

response = requests.get(url)
print("Status Code:",response.status_code)
print("Raw CSV Data:")
print(response.text[:100])

df5_macro = pd.read_csv(io.StringIO(response.text))
df5_macro = df5_macro.rename(columns={"time": "timestamp"})
df5_macro = df5_macro.rename(columns={"value": function})
df5_macro["timestamp"] = pd.to_datetime(df5_macro["timestamp"])
df5_macro = df5_macro.set_index("timestamp")
print(df5_macro.head())
print(df5_macro.sort_index(ascending=True).head())

Status Code: 200
Raw CSV Data:
timestamp,value
2025-11-20,4.1
2025-11-19,4.13
2025-11-18,4.12
2025-11-17,4.13
2025-11-14,4.14
           TREASURY_YIELD
timestamp                
2025-11-20            4.1
2025-11-19           4.13
2025-11-18           4.12
2025-11-17           4.13
2025-11-14           4.14
           TREASURY_YIELD
timestamp                
1962-01-02           4.06
1962-01-03           4.03
1962-01-04           3.99
1962-01-05           4.02
1962-01-08           4.03


### CPI

In [60]:
function= "CPI"

interval = "monthly"

url = (f"{baseUrl}"
        f"function={function}&interval={interval}&datatype={datatype}&apikey={alphaapikey}")

response = requests.get(url)
print("Status Code:",response.status_code)
print("Raw CSV Data:")
print(response.text[:100])

df6_macro = pd.read_csv(io.StringIO(response.text))
df6_macro = df6_macro.rename(columns={"time": "timestamp"})
df6_macro = df6_macro.rename(columns={"value": function})
df6_macro["timestamp"] = pd.to_datetime(df6_macro["timestamp"])
df6_macro = df6_macro.set_index("timestamp")
print(df6_macro.head())
print(df6_macro.sort_index(ascending=True).head())

Status Code: 200
Raw CSV Data:
timestamp,value
2025-09-01,324.800
2025-08-01,323.976
2025-07-01,323.048
2025-06-01,322.561
202
                CPI
timestamp          
2025-09-01  324.800
2025-08-01  323.976
2025-07-01  323.048
2025-06-01  322.561
2025-05-01  321.465
            CPI
timestamp      
1913-01-01  9.8
1913-02-01  9.8
1913-03-01  9.8
1913-04-01  9.8
1913-05-01  9.7


## yfinance API Calls

### yfinance

In [61]:
indices = ["^FTSE","^STOXX50E","^GSPC"]


df7_macro = yf.download(
    tickers=indices,
    period="max",
    interval="1d",
    auto_adjust=True
)["Close"]

# Rename to macro-style column names
df7_macro.rename(columns={
    "^FTSE": "FTSE_100",
    "^STOXX50E": "EUROSTOXX_50",
    "^GSPC": "SP500"
}, inplace=True)

print(df7_macro.head())
df7_macro.sort_index(ascending=False).head()


df7_macro.index = pd.to_datetime(df7_macro.index)
df7_macro.index.name = "timestamp"

print(df7_macro.sort_index(ascending=False).head())
print(df7_macro.sort_index(ascending=True).head())


[*********************100%***********************]  3 of 3 completed

Ticker      FTSE_100      SP500  EUROSTOXX_50
Date                                         
1927-12-30       NaN  17.660000           NaN
1928-01-03       NaN  17.760000           NaN
1928-01-04       NaN  17.719999           NaN
1928-01-05       NaN  17.549999           NaN
1928-01-06       NaN  17.660000           NaN
Ticker         FTSE_100        SP500  EUROSTOXX_50
timestamp                                         
2025-11-24  9534.910156          NaN   5528.669922
2025-11-21  9539.700195  6602.990234   5515.089844
2025-11-20  9527.700195  6538.759766   5569.919922
2025-11-19  9507.400391  6642.160156   5542.049805
2025-11-18  9552.299805  6617.319824   5534.709961
Ticker      FTSE_100      SP500  EUROSTOXX_50
timestamp                                    
1927-12-30       NaN  17.660000           NaN
1928-01-03       NaN  17.760000           NaN
1928-01-04       NaN  17.719999           NaN
1928-01-05       NaN  17.549999           NaN
1928-01-06       NaN  17.660000           NaN

## FRED API Calls

### Fred API Setup

In [62]:
fred = Fred(api_key=fredapikey)
df_macro = df_finance

### Fred API Combined API Call

In [63]:
series_map = {
    "VIX": "VIXCLS",
    #"S&P500": "SP500",
    "CHFUSD": "DEXSZUS",
    "EURUSD": "DEXUSEU",
    "US1Y": "DGS1",
    "US2Y": "DGS2",
    "US5Y": "DGS5",
    "US10Y": "DGS10",
    "US30Y": "DGS30", 
    "YC_Slope": "T10Y2Y",
    "Stress Index": "STLFSI4",
    "FEDFUNDS": "FEDFUNDS",
    "UNRATE": "UNRATE",
    "Median CPI": "MEDCPIM158SFRBCLE",
    "UMCSENT": "UMCSENT",
    "USEPUINDXD": "USEPUINDXD",
}

macro_fred = pd.DataFrame()

for name, sid in series_map.items():
    s = fred.get_series(sid)
    s.index = pd.to_datetime(s.index)
    macro_fred[name] = s

macro_fred.index.name = "timestamp"
macro_fred = macro_fred.sort_index()

df8_macro = macro_fred


# # 1) Resample to daily
# macro_daily = macro_fred.resample("D").ffill()  # carry last known value forward

# # 2) Shift by 1 day so you only use info that was available at t-1
# macro_avail = pd.DataFrame(index=macro_daily.index)

# for name in series_map.keys():
#     macro_avail[f"{name}_avail"] = macro_daily[name].shift(1)

# # shift for information availability (no look-ahead)
# for name in series_map.keys():
#     macro_fred = macro_fred.interpolate(method="time")
#     macro_fred[f"{name}_avail"] = macro_fred[name].shift(1)

# # resample to your frequency (e.g. 60min or daily)
# macro_1d = macro_fred[[f"{name}_avail" for name in series_map.keys()]]
# # macro_1d = macro_1d.interpolate(method="time") # .resample("D").ffill()  

# # merge with your main data
# # data = data.merge(macro_1d, left_index=True, right_index=True, how="left")
# macro_1d.head()
# macro_1d.sort_index(ascending=True).head()


## DF merge

In [64]:
dfs = [df1_macro, df2_macro, df3_macro, df4_macro, df5_macro, df6_macro, df7_macro, df8_macro]

#df_macro = pd.DataFrame(index=df_finance.index)
df_macro = dfs[0]

for df in dfs[1:]:
#for df in dfs:
    df_macro = df_macro.merge(df, on="timestamp", how="outer")

df_macro.sort_index(ascending=False).head(365)

,FEDERAL_FUNDS_RATE,INFLATION,REAL_GDP,REAL_GDP_PER_CAPITA,TREASURY_YIELD,CPI,FTSE_100,SP500,EUROSTOXX_50,VIX,...,US5Y,US10Y,US30Y,YC_Slope,Stress Index,FEDFUNDS,UNRATE,Median CPI,UMCSENT,USEPUINDXD
timestamp,,,,,,,,,,,,,,,,,,,,,
2025-11-24,NaN,NaN,NaN,NaN,NaN,NaN,9534.910156,NaN,5528.669922,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2025-11-21,NaN,NaN,NaN,NaN,NaN,NaN,9539.700195,6602.990234,5515.089844,23.43,...,NaN,NaN,NaN,0.55,NaN,NaN,NaN,NaN,NaN,272.48
2025-11-20,3.88,NaN,NaN,NaN,4.1,NaN,9527.700195,6538.759766,5569.919922,26.42,...,3.68,4.10,4.73,0.55,NaN,NaN,NaN,NaN,NaN,326.50
2025-11-19,3.88,NaN,NaN,NaN,4.13,NaN,9507.400391,6642.160156,5542.049805,23.66,...,3.71,4.13,4.75,0.55,NaN,NaN,NaN,NaN,NaN,214.18
2025-11-18,3.88,NaN,NaN,NaN,4.12,NaN,9552.299805,6617.319824,5534.709961,24.69,...,3.70,4.12,4.74,0.54,NaN,NaN,NaN,NaN,NaN,617.56
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2024-11-27,4.58,NaN,NaN,NaN,4.25,NaN,8274.799805,5998.740234,4733.149902,14.10,...,4.11,4.25,4.44,0.06,NaN,NaN,NaN,NaN,NaN,132.14
2024-11-26,4.58,NaN,NaN,NaN,4.3,NaN,8258.599609,6021.629883,4761.990234,14.10,...,4.17,4.30,4.48,0.09,NaN,NaN,NaN,NaN,NaN,150.15
2024-11-25,4.58,NaN,NaN,NaN,4.27,NaN,8291.700195,5987.370117,4799.870117,14.60,...,4.17,4.27,4.45,0.06,NaN,NaN,NaN,NaN,NaN,87.75


## Imputation

In [ ]:
# Ensure datetime index and sorted
df_macro.index = pd.to_datetime(df_macro.index)
df_macro = df_macro.sort_index()

# Convert all columns to numeric safely
df_macro = df_macro.apply(pd.to_numeric, errors="coerce")

# Reindex to daily grid
full_index = pd.date_range(
    start=df_macro.index.min(),
    end=df_macro.index.max(),
    freq="D"
)

df_macro = df_macro.reindex(full_index)
df_macro.index.name = "timestamp"

# Safe imputation (no future leakage)
df_macro = df_macro.ffill()
df_macro = df_macro.shift(1) # Should be done for each macroeconomic variable individualy to adjust for publication lag
#df_macro = df_macro.interpolate(method="time")
df_macro.sort_index(ascending=False).head(365)

,FEDERAL_FUNDS_RATE,INFLATION,REAL_GDP,REAL_GDP_PER_CAPITA,TREASURY_YIELD,CPI,FTSE_100,SP500,EUROSTOXX_50,VIX,...,US5Y,US10Y,US30Y,YC_Slope,Stress Index,FEDFUNDS,UNRATE,Median CPI,UMCSENT,USEPUINDXD
timestamp,,,,,,,,,,,,,,,,,,,,,
2025-11-24,3.88,2.949525,5943.384000,69499.000000,4.100,324.800000,9534.910156,6602.990234,5528.669922,23.43,...,3.68,4.100,4.73,0.550,-0.506400,4.090000,4.400000,2.384737,53.600000,272.48
2025-11-23,3.88,2.949525,5943.384000,69499.000000,4.100,324.800000,9536.506836,6602.990234,5524.143229,23.43,...,3.68,4.100,4.73,0.550,-0.506400,4.090000,4.400000,2.384737,53.600000,272.48
2025-11-22,3.88,2.949525,5943.384000,69499.000000,4.100,324.800000,9538.103516,6602.990234,5519.616536,23.43,...,3.68,4.100,4.73,0.550,-0.506400,4.090000,4.400000,2.384737,53.600000,272.48
2025-11-21,3.88,2.949525,5943.384000,69499.000000,4.100,324.800000,9539.700195,6602.990234,5515.089844,23.43,...,3.68,4.100,4.73,0.550,-0.506400,4.090000,4.400000,2.384737,53.600000,272.48
2025-11-20,3.88,2.949525,5943.384000,69499.000000,4.100,324.800000,9527.700195,6538.759766,5569.919922,26.42,...,3.68,4.100,4.73,0.550,-0.506400,4.090000,4.400000,2.384737,53.600000,326.50
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2024-11-29,4.58,2.949525,5855.802043,69008.380435,4.180,315.597533,8287.299805,6032.379883,4804.399902,13.51,...,4.05,4.180,4.36,0.050,-0.618000,4.497705,4.108197,3.549274,71.754098,174.81
2024-11-28,4.58,2.949525,5858.198348,69010.543478,4.215,315.593800,8281.200195,6015.560059,4758.649902,13.90,...,4.08,4.215,4.40,0.055,-0.616971,4.502787,4.111475,3.537731,71.755738,162.72
2024-11-27,4.58,2.949525,5860.594652,69012.706522,4.250,315.590067,8274.799805,5998.740234,4733.149902,14.10,...,4.11,4.250,4.44,0.060,-0.615943,4.507869,4.114754,3.526188,71.757377,132.14


In [66]:
df_macro.isna().sum()

FEDERAL_FUNDS_RATE     15156
INFLATION              17166
REAL_GDP               32507
REAL_GDP_PER_CAPITA    12418
TREASURY_YIELD         17898
CPI                        0
FTSE_100               25934
SP500                   5476
EUROSTOXX_50           34421
VIX                    28125
CHFUSD                 28125
EURUSD                 31414
US1Y                   28125
US2Y                   28125
US5Y                   28125
US10Y                  28125
US30Y                  28125
YC_Slope               28125
Stress Index           29584
FEDFUNDS               28155
UNRATE                 28155
Median CPI             28155
UMCSENT                28155
USEPUINDXD             28125
dtype: int64

In [67]:
df_macro.loc[:, df_macro.isna().any(axis=0)].head(132)

,FEDERAL_FUNDS_RATE,INFLATION,REAL_GDP,REAL_GDP_PER_CAPITA,TREASURY_YIELD,FTSE_100,SP500,EUROSTOXX_50,VIX,CHFUSD,...,US5Y,US10Y,US30Y,YC_Slope,Stress Index,FEDFUNDS,UNRATE,Median CPI,UMCSENT,USEPUINDXD
timestamp,,,,,,,,,,,,,,,,,,,,,
1913-01-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1913-01-02,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1913-01-03,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1913-01-04,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1913-01-05,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1913-05-08,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1913-05-09,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1913-05-10,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Data merge

In [68]:
data = data.drop(columns=df_macro.columns.intersection(data.columns))

data = data.merge(df_macro, left_index=True, right_index=True, how="left")
data.sort_index(ascending=False).head()

,target_vol_next_1d,open,high,low,close,volume,SMA,EMA,RSI,ATR,...,US5Y,US10Y,US30Y,YC_Slope,Stress Index,FEDFUNDS,UNRATE,Median CPI,UMCSENT,USEPUINDXD
timestamp,,,,,,,,,,,,,,,,,,,,,
2025-11-20,0.014759,37.71,38.1400,36.78,36.79,2701233,38.2100,37.9561,34.5436,0.6924,...,3.68,4.10,4.73,0.55,-0.5064,4.09,4.4,2.384737,53.6,326.50
2025-11-19,0.036309,38.11,38.2650,37.91,38.20,1307518,38.6783,38.4226,44.3556,0.6364,...,3.71,4.13,4.75,0.55,-0.5064,4.09,4.4,2.384737,53.6,214.18
2025-11-18,0.009321,37.81,38.2100,37.73,38.03,1979153,38.8083,38.5116,42.5280,0.6581,...,3.70,4.12,4.74,0.54,-0.5064,4.09,4.4,2.384737,53.6,617.56
2025-11-17,0.012642,38.79,38.8851,38.11,38.25,2165469,38.8750,38.7043,44.2755,0.6687,...,3.72,4.13,4.73,0.53,-0.5064,4.09,4.4,2.384737,53.6,585.16
2025-11-14,0.020134,38.63,39.0400,38.49,38.92,1545463,38.8850,38.8860,50.0969,0.6578,...,3.74,4.14,4.74,0.52,-0.5064,4.09,4.4,2.384737,53.6,307.59


# Drop Na

In [36]:
data = data.dropna()
data = data.copy()
print(len(data))
print(data.columns)
data.head()

2750
Index(['ctc_vol', 'parkinson_vol', 'ctc_vol_shifted', 'parkinson_vol_shifted',
       'open', 'high', 'low', 'close', 'volume', 'SMA', 'EMA', 'RSI', 'ATR',
       'Real Lower Band', 'Real Middle Band', 'Real Upper Band', 'return',
       'log_return', 'abs_log_return', 'range_t', 'body', 'upper_wick',
       'lower_wick', 'body_ratio', 'upper_wick_ratio', 'lower_wick_ratio',
       'bb_width', 'bb_width_norm', 'day_of_week'],
      dtype='object')


,ctc_vol,parkinson_vol,ctc_vol_shifted,parkinson_vol_shifted,open,high,low,close,volume,SMA,...,range_t,body,upper_wick,lower_wick,body_ratio,upper_wick_ratio,lower_wick_ratio,bb_width,bb_width_norm,day_of_week
timestamp,,,,,,,,,,,,,,,,,,,,,
2014-12-19,0.011974,0.012418,0.010055,0.010384,17.33,17.4466,17.270,17.38,1021196,12.1904,...,0.010174,0.05,0.0666,0.060,0.283126,0.377123,0.339751,3.4782,0.275455,4
2014-12-22,0.010055,0.010384,0.010383,0.008574,17.45,17.5400,17.445,17.53,701889,12.2056,...,0.005431,0.08,0.0100,0.005,0.842105,0.105263,0.052632,3.4699,0.274691,0
2014-12-23,0.010383,0.008574,0.010120,0.006590,17.50,17.6100,17.470,17.50,1097125,12.2431,...,0.007982,0.00,0.1100,0.030,0.000000,0.785714,0.214286,3.4740,0.275077,1
2014-12-24,0.010120,0.006590,0.008407,0.005599,17.47,17.6300,17.460,17.61,450226,12.2887,...,0.009689,0.14,0.0200,0.010,0.823529,0.117647,0.058824,3.4677,0.274482,2
2014-12-26,0.008407,0.005599,0.011705,0.005587,17.52,17.7200,17.510,17.67,531820,12.3238,...,0.011922,0.15,0.0500,0.010,0.714286,0.238095,0.047619,0.8657,0.069593,4


In [37]:
data.sort_index(ascending=False).head()

,ctc_vol,parkinson_vol,ctc_vol_shifted,parkinson_vol_shifted,open,high,low,close,volume,SMA,...,range_t,body,upper_wick,lower_wick,body_ratio,upper_wick_ratio,lower_wick_ratio,bb_width,bb_width_norm,day_of_week
timestamp,,,,,,,,,,,,,,,,,,,,,
2025-11-25,0.022277,0.011813,0.023114,0.011973,37.29,37.640,37.0950,37.59,1350342,37.4117,...,0.014585,0.30,0.050,0.1950,0.550459,0.091743,0.357798,2.7982,0.073352,1
2025-11-24,0.017886,0.012564,0.022277,0.011813,36.61,36.895,36.4435,36.79,1973760,37.5217,...,0.012313,0.18,0.105,0.1665,0.398671,0.232558,0.368771,2.8638,0.074924,0
2025-11-21,0.018387,0.012516,0.017886,0.012564,37.04,37.200,36.6550,37.07,1989950,37.8767,...,0.014759,0.03,0.130,0.3850,0.055046,0.238532,0.706422,2.5580,0.066766,4
2025-11-20,0.016286,0.009474,0.018387,0.012516,37.71,38.140,36.7800,36.79,2701233,38.2100,...,0.036309,0.92,0.430,0.0100,0.676471,0.316176,0.007353,2.3108,0.060239,3
2025-11-19,0.008543,0.009376,0.016286,0.009474,38.11,38.265,37.9100,38.20,1307518,38.6783,...,0.009321,0.09,0.065,0.2000,0.253521,0.183099,0.563380,1.8168,0.047273,2


# Write to .csv

In [38]:
data.to_csv(f"data_{symbol}.csv", index=True)